# Regresión de carga de calefacción

[Abrir y ejecutar en Google Colab](https://colab.research.google.com/github/al073016/Practica-4-Modelos-de-machine-learning/blob/main/02_Regresion_Energetica.ipynb)

Pregunta: ¿qué carga de calefacción requiere un edificio según su diseño? Fuente: [Energy Efficiency, UCI](https://archive.ics.uci.edu/dataset/242/energy+efficiency), licencia CC BY 4.0. Los 768 casos son configuraciones **simuladas** de edificios; esto limita la generalización a edificios reales. Objetivo: `Y1` (Heating Load), en las unidades del dataset.

**Uso:** abre este notebook en Google Colab y elige **Entorno de ejecución → Ejecutar todo**. Las celdas descargan los datos automáticamente. La última celda permite probar una muestra del conjunto de prueba. El entrenamiento en CPU puede tardar varios minutos.

## 1. Bibliotecas y descarga automática

In [ ]:
import importlib.util, subprocess, sys
needed = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
          'sklearn': 'scikit-learn', 'tensorflow': 'tensorflow', 'openpyxl': 'openpyxl'}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SEED = 42
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
import io, zipfile, urllib.request
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

url = 'https://archive.ics.uci.edu/static/public/242/energy+efficiency.zip'
blob = urllib.request.urlopen(url, timeout=60).read()
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    files = [n for n in z.namelist() if n.lower().endswith('.xlsx')]
    assert files, z.namelist()
    df = pd.read_excel(io.BytesIO(z.read(files[0])), engine='openpyxl')
df = df.iloc[:, :10].copy()
df.columns = [f'X{i}' for i in range(1,9)] + ['Y1','Y2']
df = df.dropna(subset=['Y1'])
assert len(df) == 768
print(df.shape, '\nFaltantes:', df.isna().sum().sum())
display(df.head())

## 2. División y escalado
60 % entrenamiento, 20 % validación, 20 % prueba. Los escaladores solo aprenden de entrenamiento.

In [ ]:
X = df[[f'X{i}' for i in range(1,9)]].to_numpy(dtype='float32')
y = df['Y1'].to_numpy(dtype='float32').reshape(-1,1)
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.25, random_state=SEED)
x_scaler, y_scaler = StandardScaler(), StandardScaler()
X_train_p = x_scaler.fit_transform(X_train).astype('float32')
X_val_p = x_scaler.transform(X_val).astype('float32')
X_test_p = x_scaler.transform(X_test).astype('float32')
y_train_p = y_scaler.fit_transform(y_train).astype('float32')
y_val_p = y_scaler.transform(y_val).astype('float32')
print('Tamaños:', len(X_train), len(X_val), len(X_test))

## 3. Ajuste de arquitectura
Comparamos dos tamaños de MLP y escogemos el menor MAE de validación, medido en unidades originales.

In [ ]:
def build_model(widths):
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(8,))])
    for width in widths:
        model.add(tf.keras.layers.Dense(width, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='linear'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

trials = []
for widths in [(32, 16), (64, 32)]:
    tf.keras.utils.set_random_seed(SEED)
    model = build_model(widths)
    history = model.fit(X_train_p, y_train_p, validation_data=(X_val_p, y_val_p),
                        epochs=300, batch_size=16, verbose=0,
                        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True),
                                   tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=10, factor=0.5)])
    pred_val = y_scaler.inverse_transform(model.predict(X_val_p, verbose=0)).ravel()
    mae = mean_absolute_error(y_val.ravel(), pred_val)
    trials.append((mae, widths, model, history))
    print(widths, 'épocas:', len(history.history['loss']), 'MAE validación:', round(mae, 3))

best_mae, best_widths, model, history = min(trials, key=lambda t: t[0])
print('Modelo elegido:', best_widths)

## 4. Evaluación final en prueba
Se compara con una predicción ingenua: la media de calefacción del entrenamiento.

In [ ]:
pred = y_scaler.inverse_transform(model.predict(X_test_p, verbose=0)).ravel()
actual = y_test.ravel()
baseline = np.full_like(actual, y_train.mean())
for name, values in [('Media de entrenamiento', baseline), ('Red neuronal', pred)]:
    print(name, 'MAE:', round(mean_absolute_error(actual, values), 3),
          'RMSE:', round(np.sqrt(mean_squared_error(actual, values)), 3),
          'R²:', round(r2_score(actual, values), 4))
fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].plot(history.history['loss'], label='Entrenamiento')
axes[0].plot(history.history['val_loss'], label='Validación')
axes[0].set_title('Pérdida MSE escalada'); axes[0].legend()
axes[1].scatter(actual, pred, alpha=0.6)
lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
axes[1].plot(lims, lims, 'k--'); axes[1].set(xlabel='Real', ylabel='Predicho', title='Carga de calefacción')
plt.tight_layout(); plt.show()

## 5. Probar una predicción
Modifica `indice` para inspeccionar otro edificio del conjunto de prueba.

In [ ]:
indice = 0
assert 0 <= indice < len(X_test)
print('Características:', dict(zip([f'X{i}' for i in range(1,9)], X_test[indice].tolist())))
print('Predicción:', round(float(pred[indice]), 2), '| Real:', round(float(actual[indice]), 2))